In [1]:
import os
from dotenv import load_dotenv
import gradio as gr
import google.generativeai

In [2]:
load_dotenv()
google_api_key = os.getenv('GEMINI_API_KEY')

In [3]:
google.generativeai.configure()

In [4]:
def stream_gemini(prompt):
    genai.configure(api_key='google_api_key')
    model = genai.GenerativeModel('gemini-pro',system_instruction=[system_message])
    
    response = model.generate_content(
        prompt, 
        stream=True
    )
    
    full_response = ""
    for chunk in response:
        full_response += chunk.text or ""
        yield full_response

In [10]:
system_message = "you are an medical AI assistant, and you need to give result of reaction between compounds and medicine an it's effect on human body"

In [ ]:
user_prompt = "find the reaction between the medicine in the list:"
user_prompt += json.dumps(medicine_list)

In [6]:
class medicineCollector:
    def __init__(self):
        self.user_strings = []

    def collect_strings(self):
        while True:
            user_input = input("Enter a medicine: ").strip()
            if user_input.lower() == '':  
                break
            if user_input in self.user_strings:
                pass
            self.user_strings.append(user_input)
            print(f"Current list: {self.user_strings}")
        print("Final list of strings:", self.user_strings)
        return self.user_strings

# Self-invoking the function
if __name__ == "__main__":
    medicineCollector().collect_strings()

Current list: ['paracetamol']
Current list: ['paracetamol', 'asprine']
Current list: ['paracetamol', 'asprine', 'alcohol']
Final list of strings: ['paracetamol', 'asprine', 'alcohol']


In [7]:
def check_medicine_interactions(StringCollector):
    prompt = f"Please check the interactions between the following medicines: \n"
    prompt += collect_strings()
    result = stream_gemini(prompt)
    yield from result

In [9]:
def create_medicine_interface():
    """Create Gradio interface for medicine interaction tracking"""
    with gr.Blocks() as demo:
        with gr.Row():
            medicine_input = gr.Textbox(label="Enter Medicine Name")
            medicine_list = gr.Dataframe(headers=["Medicines"], type="array")
        
        with gr.Row():
            add_btn = gr.Button("Add Medicine")
            clear_btn = gr.Button("Clear All")
            check_btn = gr.Button("Check Interactions")
        
        output = gr.Textbox(label="Interaction Results")
        
        # Add medicine to list
        def add_medicine_to_list(input_med, current_list):
            if not input_med.strip():  # Check if input is empty
                return "", current_list
            # Check if medicine already exists
            if input_med.strip() not in [med[0] for med in current_list]:
                return "", current_list + [[input_med.strip()]]
            return "", current_list

        add_btn.click(
            fn=add_medicine_to_list,
            inputs=[medicine_input, medicine_list],
            outputs=[medicine_input, medicine_list]
        )
        
        # Clear all medicines
        clear_btn.click(
            fn=lambda: [], 
            outputs=[medicine_list]
        )
        
        # Check interactions
        check_btn.click(
            fn=lambda x: check_medicine_interactions([med[0] for med in x]), 
            inputs=[medicine_list], 
            outputs=[output]
        )
    
    return demo

iface = create_medicine_interface()
iface.launch()


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [8]:
# def create_medicine_interface():
#     """Create Gradio interface for medicine interaction tracking"""
#     with gr.Blocks() as demo:
#         with gr.Row():
#             medicine_input = gr.Textbox(label="Enter Medicine Name")
#             medicine_list = gr.Dataframe(headers=["Medicines"], type="array")
        
#         with gr.Row():
#             add_btn = gr.Button("Add Medicine")
#             clear_btn = gr.Button("Clear All")
#             check_btn = gr.Button("Check Interactions")
        
#         output = gr.Textbox(label="Interaction Results")
        
#         # Add medicine to list
#         add_btn.click(
#             lambda x, y: (
#                 "",  # Clear input box
#                 y + [[x]] if x.strip() not in [med[0] for med in y] else y
#             ),
#             inputs=[medicine_input, medicine_list],
#             outputs=[medicine_input, medicine_list]
#         )
        
#         # Store medicine array
#         def add_medicine(self, medicine):
#             medicine = medicine.strip()
#             if medicine and medicine not in self.medicine_array:
#                 self.medicine_array.append(medicine)
#             return self.medicine_array
        
#         # Clear all medicines
#         clear_btn.click(
#             lambda: [], 
#             outputs=[medicine_list]
#         )
        
#         # Check interactions
#         check_btn.click(
#             lambda x: check_medicine_interactions([med[0] for med in x]), 
#             inputs=[medicine_list], 
#             outputs=[output]
#         )
    
#     return demo

# # Launch the interface
# iface = create_medicine_interface()
# iface.launch()

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
